# Session-based Binary Prediction with XLNet — CPU

Replicates the Transformers4Rec XLNet architecture using only `transformers==4.30.1` and `torch` (CPU only), modified for binary classification.  
No dependency on transformers4rec or merlin.

In [ ]:
import os
import gc
import csv
import glob
import math
import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR

from transformers import XLNetConfig, XLNetModel
from tqdm.auto import tqdm

device = torch.device("cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Configuration

In [ ]:
# ---------------------------------------------------------------------------
# Feature configuration
# ---------------------------------------------------------------------------
CATEGORICAL_FEATURES = {
    "sess_pid_seq":  {"cardinality": 164386, "embedding_dim": 64},
    "sess_csid_seq": {"cardinality": 622,    "embedding_dim": 32},
    "sess_ccid_seq": {"cardinality": 129,    "embedding_dim": 16},
    "sess_bid_seq":  {"cardinality": 3426,   "embedding_dim": 32},
}

CONTINUOUS_FEATURES = [
    "sess_price_log_norm_seq",
    "sess_dtime_secs_log_norm_seq",
    "sess_prod_recency_days_log_norm_seq",
]

# Single target: TARGET_COLUMN = "target"
# Multi target:  TARGET_COLUMN = ["target_purchase", "target_cart"]
TARGET_COLUMN = "target"

# Derived — used by dataset, model, and inference cells
TARGET_COLUMNS = TARGET_COLUMN if isinstance(TARGET_COLUMN, list) else [TARGET_COLUMN]
NUM_CLASSES = len(TARGET_COLUMNS)

# ---------------------------------------------------------------------------
# Model hyperparameters
# ---------------------------------------------------------------------------
MAX_SEQ_LEN = 20
D_MODEL = 448
N_HEAD = 8
N_LAYER = 2
CONTINUOUS_PROJ_DIM = 64
CLASSIFIER_HIDDEN_DIM = 128

# ---------------------------------------------------------------------------
# Training hyperparameters
# ---------------------------------------------------------------------------
LEARNING_RATE = 0.00020171456712823088
WEIGHT_DECAY = 2.747484129693843e-05
NUM_EPOCHS = 10
TRAIN_BATCH_SIZE = 256
EVAL_BATCH_SIZE = 512

# ---------------------------------------------------------------------------
# Data paths
# ---------------------------------------------------------------------------
SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "preprocessed_data"))
OUTPUT_DIR = os.path.abspath(os.path.join(os.getcwd(), "data_trimmed"))

START_WINDOW = int(os.environ.get("start_window_index", "1"))
FINAL_WINDOW = int(os.environ.get("final_window_index", "30"))

print(f"Source data:  {SRC_DIR}")
print(f"Trimmed data: {OUTPUT_DIR}")
print(f"Window range: {START_WINDOW} \u2013 {FINAL_WINDOW}")
print(f"Targets ({NUM_CLASSES}): {TARGET_COLUMNS}")

## Dataset

In [ ]:
class SessionDataset(Dataset):
    """Loads parquet files and converts variable-length sessions into
    fixed-length tensors (padded / truncated to MAX_SEQ_LEN)."""

    def __init__(self, parquet_paths, max_seq_len=MAX_SEQ_LEN):
        dfs = [pd.read_parquet(p) for p in parquet_paths]
        self.df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq_len = min(int(row["sess_seq_len"]), self.max_seq_len)
        result = {}

        for feat_name in CATEGORICAL_FEATURES:
            arr = np.array(row[feat_name], dtype=np.int64)[: self.max_seq_len]
            padded = np.zeros(self.max_seq_len, dtype=np.int64)
            padded[: len(arr)] = arr
            result[feat_name] = torch.tensor(padded, dtype=torch.long)

        cont_arrays = []
        for feat_name in CONTINUOUS_FEATURES:
            arr = np.array(row[feat_name], dtype=np.float32)[: self.max_seq_len]
            padded = np.zeros(self.max_seq_len, dtype=np.float32)
            padded[: len(arr)] = arr
            cont_arrays.append(padded)
        result["continuous"] = torch.tensor(
            np.stack(cont_arrays, axis=-1), dtype=torch.float32
        )

        mask = np.zeros(self.max_seq_len, dtype=np.float32)
        mask[:seq_len] = 1.0
        result["attention_mask"] = torch.tensor(mask, dtype=torch.float32)

        if NUM_CLASSES > 1:
            result["target"] = torch.tensor(
                [float(row[c]) for c in TARGET_COLUMNS], dtype=torch.float32
            )  # (NUM_CLASSES,)
        else:
            result["target"] = torch.tensor(
                float(row[TARGET_COLUMNS[0]]), dtype=torch.float32
            )  # scalar

        return result


_sample_paths = glob.glob(os.path.join(OUTPUT_DIR, "0001/train.parquet"))
_ds = SessionDataset(_sample_paths)
_item = _ds[0]
print(f"Dataset size: {len(_ds)}")
for k, v in _item.items():
    print(f"  {k}: {v.shape} {v.dtype}")
del _ds, _item

## Model Architecture

In [ ]:
class SessionXLNetModel(nn.Module):
    """XLNet-based session model with a binary/multi-label classification head.
    Pools the XLNet output over valid positions, then feeds through
    an MLP classifier → sigmoid for prediction."""

    def __init__(self):
        super().__init__()

        self.embeddings = nn.ModuleDict()
        for feat, cfg in CATEGORICAL_FEATURES.items():
            self.embeddings[feat] = nn.Embedding(
                cfg["cardinality"], cfg["embedding_dim"], padding_idx=0
            )

        self.continuous_proj = nn.Sequential(
            nn.Linear(len(CONTINUOUS_FEATURES), CONTINUOUS_PROJ_DIM),
            nn.ReLU(),
        )

        cat_dim = sum(c["embedding_dim"] for c in CATEGORICAL_FEATURES.values())
        total_input_dim = cat_dim + CONTINUOUS_PROJ_DIM
        self.input_mlp = nn.Sequential(
            nn.Linear(total_input_dim, D_MODEL),
            nn.ReLU(),
        )

        xlnet_config = XLNetConfig(
            vocab_size=2,
            d_model=D_MODEL,
            n_layer=N_LAYER,
            n_head=N_HEAD,
            d_inner=4 * D_MODEL,
            ff_activation="gelu",
            attn_type="bi",
            mem_len=0,
            dropout=0.0,
        )
        self.xlnet = XLNetModel(xlnet_config)

        # Classification head: outputs NUM_CLASSES logits
        self.classifier = nn.Sequential(
            nn.Linear(D_MODEL, CLASSIFIER_HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(CLASSIFIER_HIDDEN_DIM, NUM_CLASSES),
        )

        self.loss_fn = nn.BCEWithLogitsLoss()

    def forward(self, batch):
        cat_embeds = [self.embeddings[f](batch[f]) for f in CATEGORICAL_FEATURES]
        cont_proj = self.continuous_proj(batch["continuous"])

        x = torch.cat(cat_embeds + [cont_proj], dim=-1)
        x = self.input_mlp(x)

        attention_mask = batch["attention_mask"]

        xlnet_out = self.xlnet(inputs_embeds=x, attention_mask=attention_mask)
        hidden = xlnet_out.last_hidden_state  # (B, T, D_MODEL)

        # Masked mean pooling over valid positions
        mask_expanded = attention_mask.unsqueeze(-1)  # (B, T, 1)
        hidden_masked = hidden * mask_expanded
        pooled = hidden_masked.sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)  # (B, D_MODEL)

        logits = self.classifier(pooled)  # (B, NUM_CLASSES)
        if NUM_CLASSES == 1:
            logits = logits.squeeze(-1)   # (B,) for single target
        targets = batch["target"]
        loss = self.loss_fn(logits, targets)

        return loss, logits


model = SessionXLNetModel().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

## Evaluation Metrics (Binary Classification)

In [ ]:
from sklearn.metrics import roc_auc_score


def compute_auc(logits, targets):
    """Compute AUC from raw logits and targets.
    Returns a dict of per-target AUCs when NUM_CLASSES > 1, or a single float.
    """
    probs = torch.sigmoid(logits).cpu().numpy()
    targets_np = targets.cpu().numpy()

    if NUM_CLASSES > 1:
        auc_dict = {}
        for i, col in enumerate(TARGET_COLUMNS):
            try:
                auc_dict[col] = roc_auc_score(targets_np[:, i], probs[:, i])
            except ValueError:
                auc_dict[col] = float("nan")
        return auc_dict
    else:
        try:
            return roc_auc_score(targets_np, probs)
        except ValueError:
            return float("nan")

## Training & Evaluation Helpers

In [ ]:
# ---------------------------------------------------------------------------
# CSV logger
# ---------------------------------------------------------------------------
LOG_FIELDS = [
    "timestamp", "phase", "day", "epoch", "batch", "loss", "auc",
]


def init_log(path):
    with open(path, "w", newline="") as f:
        csv.writer(f).writerow(LOG_FIELDS)
    return path


def log_row(path, **kwargs):
    with open(path, "a", newline="") as f:
        csv.writer(f).writerow([kwargs.get(c, "") for c in LOG_FIELDS])


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
def train_one_day(model, train_loader, optimizer, scheduler, log_path, day):
    model.train()
    epoch_bar = tqdm(range(NUM_EPOCHS), desc="Epochs", unit="epoch")
    for epoch in epoch_bar:
        epoch_loss = 0.0
        n_batches = 0
        batch_bar = tqdm(train_loader, desc=f"  Epoch {epoch+1}/{NUM_EPOCHS}",
                         leave=False, unit="batch")
        for batch in batch_bar:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            loss, _ = model(batch)
            loss.backward()
            optimizer.step()
            scheduler.step()

            batch_loss = loss.item()
            epoch_loss += batch_loss
            n_batches += 1
            batch_bar.set_postfix(batch_loss=f"{batch_loss:.4f}")

            log_row(log_path, timestamp=datetime.datetime.now().isoformat(),
                    phase="train", day=day, epoch=epoch + 1,
                    batch=n_batches, loss=f"{batch_loss:.6f}")

        avg_epoch_loss = epoch_loss / max(n_batches, 1)
        epoch_bar.set_postfix(epoch_loss=f"{avg_epoch_loss:.4f}")
        print(f"    Epoch {epoch+1}/{NUM_EPOCHS}  loss={avg_epoch_loss:.4f}")


# ---------------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------------
def evaluate(model, eval_loader, log_path=None, day=None):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    all_logits = []
    all_targets = []

    with torch.no_grad():
        for batch in tqdm(eval_loader, desc="  Evaluating", leave=False, unit="batch"):
            batch = {k: v.to(device) for k, v in batch.items()}
            loss, logits = model(batch)

            total_loss += loss.item()
            n_batches += 1
            all_logits.append(logits.cpu())
            all_targets.append(batch["target"].cpu())

    all_logits = torch.cat(all_logits)
    all_targets = torch.cat(all_targets)
    auc = compute_auc(all_logits, all_targets)
    avg_loss = total_loss / max(n_batches, 1)

    metrics = {"loss": avg_loss}
    if isinstance(auc, dict):
        for col, val in auc.items():
            metrics[f"auc_{col}"] = val
        metrics["auc_mean"] = np.nanmean(list(auc.values()))
    else:
        metrics["auc"] = auc

    if log_path is not None:
        auc_str = (f"{metrics.get('auc_mean', metrics.get('auc', float('nan'))):.6f}")
        log_row(log_path,
                timestamp=datetime.datetime.now().isoformat(),
                phase="eval", day=day,
                loss=f"{avg_loss:.6f}", auc=auc_str)

    return metrics


def wipe_memory():
    gc.collect()

## Sliding-Window Training

In [ ]:
time_stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
log_path = init_log(f"training_log_cpu_{time_stamp}.csv")
print(f"Logging to: {log_path}")

for time_index in range(START_WINDOW, FINAL_WINDOW):
    train_paths = glob.glob(
        os.path.join(OUTPUT_DIR, f"{time_index:04d}/train.parquet")
    )
    eval_day = time_index + 1
    eval_paths = glob.glob(
        os.path.join(OUTPUT_DIR, f"{eval_day:04d}/valid.parquet")
    )
    if not train_paths:
        print(f"Day {time_index}: no train data, skipping.")
        continue

    print("=" * 50)
    print(f"Day {time_index}: training  (eval on day {eval_day})")
    print("=" * 50)

    train_ds = SessionDataset(train_paths)
    train_loader = DataLoader(
        train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, drop_last=False,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    total_steps = len(train_loader) * NUM_EPOCHS
    scheduler = LambdaLR(
        optimizer, lr_lambda=lambda step: max(0.0, 1 - step / max(total_steps, 1))
    )

    train_one_day(model, train_loader, optimizer, scheduler,
                  log_path, day=time_index)

    if eval_paths:
        eval_ds = SessionDataset(eval_paths)
        eval_loader = DataLoader(
            eval_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
        )
        eval_metrics = evaluate(model, eval_loader,
                                log_path=log_path, day=eval_day)
        print(f"  Eval results (day {eval_day}):")
        for k in sorted(eval_metrics):
            print(f"    {k} = {eval_metrics[k]:.6f}")
    else:
        print(f"  No eval data for day {eval_day}.")

    wipe_memory()

print("\nTraining complete.")

## Final Evaluation on Test Splits

In [ ]:
results = []

for day in range(START_WINDOW, FINAL_WINDOW + 1):
    test_paths = glob.glob(
        os.path.join(OUTPUT_DIR, f"{day:04d}/test.parquet")
    )
    if not test_paths:
        continue
    test_ds = SessionDataset(test_paths)
    test_loader = DataLoader(
        test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
    )
    metrics = evaluate(model, test_loader, log_path=log_path, day=day)
    metrics["time_index"] = day
    results.append(metrics)
    print(f"Day {day}:", {k: f"{v:.4f}" for k, v in metrics.items() if k != "time_index"})

results_df = pd.DataFrame(results).set_index("time_index")
results_df.loc["overall"] = results_df.mean()

print("\n" + "=" * 60)
print("Results Summary:")
print(results_df.to_string())

results_df.to_csv(f"evaluation_results_cpu_{time_stamp}.csv")
print(f"\nSaved to evaluation_results_cpu_{time_stamp}.csv")
print(f"Full training log: {log_path}")

## Visualization

In [ ]:
import matplotlib.pyplot as plt

df_plot = results_df.drop("overall", errors="ignore").copy()
df_plot.index = pd.to_numeric(df_plot.index)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(df_plot.index, df_plot["loss"], marker="o", markersize=5)
axes[0].set_xlabel("Day")
axes[0].set_ylabel("Loss")
axes[0].set_title("Test Loss Over Time")

axes[1].plot(df_plot.index, df_plot["auc"], marker="s", markersize=5, color="tab:orange")
axes[1].set_xlabel("Day")
axes[1].set_ylabel("AUC")
axes[1].set_title("Test AUC Over Time")

plt.suptitle("Binary Classification — CPU", fontsize=14)
plt.tight_layout()
plt.savefig(f"evaluation_metrics_over_time_cpu_{time_stamp}.png", bbox_inches="tight")
plt.show()

## Save Model

In [ ]:
model_path = os.path.join(os.getcwd(), "saved_model")
os.makedirs(model_path, exist_ok=True)
torch.save(model.state_dict(), os.path.join(model_path, "model.pt"))
print(f"Model saved to {model_path}/model.pt")

## Production-Style Inference from test.pkl

In [ ]:
import pickle

# ---------------------------------------------------------------------------
# 1. Load test.pkl
# ---------------------------------------------------------------------------
TEST_PKL_PATH = os.path.join(os.getcwd(), "test.pkl")
with open(TEST_PKL_PATH, "rb") as f:
    test_df = pickle.load(f)
print(f"Loaded {len(test_df)} sessions from {TEST_PKL_PATH}")
print(f"Target columns ({NUM_CLASSES}): {TARGET_COLUMNS}")

# ---------------------------------------------------------------------------
# 2. Build tensors for inference (same logic as SessionDataset.__getitem__)
# ---------------------------------------------------------------------------
def df_to_batch(df, max_seq_len=MAX_SEQ_LEN):
    """Convert a DataFrame of sessions into a batched tensor dict."""
    all_cat = {f: [] for f in CATEGORICAL_FEATURES}
    all_cont = []
    all_mask = []

    for _, row in df.iterrows():
        seq_len = min(int(row["sess_seq_len"]), max_seq_len)

        for feat_name in CATEGORICAL_FEATURES:
            arr = np.array(row[feat_name], dtype=np.int64)[:max_seq_len]
            padded = np.zeros(max_seq_len, dtype=np.int64)
            padded[:len(arr)] = arr
            all_cat[feat_name].append(padded)

        cont_arrays = []
        for feat_name in CONTINUOUS_FEATURES:
            arr = np.array(row[feat_name], dtype=np.float32)[:max_seq_len]
            padded = np.zeros(max_seq_len, dtype=np.float32)
            padded[:len(arr)] = arr
            cont_arrays.append(padded)
        all_cont.append(np.stack(cont_arrays, axis=-1))

        mask = np.zeros(max_seq_len, dtype=np.float32)
        mask[:seq_len] = 1.0
        all_mask.append(mask)

    batch = {}
    for feat_name in CATEGORICAL_FEATURES:
        batch[feat_name] = torch.tensor(np.stack(all_cat[feat_name]), dtype=torch.long)
    batch["continuous"] = torch.tensor(np.stack(all_cont), dtype=torch.float32)
    batch["attention_mask"] = torch.tensor(np.stack(all_mask), dtype=torch.float32)
    # Dummy target — only needed so forward() doesn't error
    batch["target"] = torch.zeros(len(df), NUM_CLASSES, dtype=torch.float32) if NUM_CLASSES > 1 \
        else torch.zeros(len(df), dtype=torch.float32)
    return batch

# ---------------------------------------------------------------------------
# 3. Run inference in mini-batches
# ---------------------------------------------------------------------------
model.eval()
all_logits_list = []

for start in tqdm(range(0, len(test_df), EVAL_BATCH_SIZE), desc="Inference"):
    end = min(start + EVAL_BATCH_SIZE, len(test_df))
    chunk_df = test_df.iloc[start:end]
    batch = df_to_batch(chunk_df)
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        _, logits = model(batch)

    all_logits_list.append(logits.cpu().numpy())

# all_logits_out shape: (N,) for single target, (N, NUM_CLASSES) for multi
all_logits_out = np.concatenate(all_logits_list).astype(np.float32)
all_probs = 1.0 / (1.0 + np.exp(-all_logits_out))  # sigmoid

# ---------------------------------------------------------------------------
# 4. Build final analysis DataFrame
# ---------------------------------------------------------------------------
inference_df = test_df.copy()

if NUM_CLASSES > 1:
    # Multi-target: one logit & prob column per target
    for i, col in enumerate(TARGET_COLUMNS):
        inference_df[f"logit_{col}"] = all_logits_out[:, i]
        inference_df[f"prob_{col}"] = all_probs[:, i]

    # Per-target AUC
    has_all_targets = all(col in test_df.columns for col in TARGET_COLUMNS)
    if has_all_targets:
        print()
        auc_results = {}
        for i, col in enumerate(TARGET_COLUMNS):
            gt = test_df[col].values.astype(np.float32)
            preds = all_probs[:, i]
            try:
                auc = roc_auc_score(gt, preds)
            except ValueError:
                auc = float("nan")
            auc_results[col] = auc
            print(f"AUC [{col}]: {auc:.6f}  |  "
                  f"pos rate (gt): {gt.mean():.4%}  |  "
                  f"pos rate (pred>0.5): {(preds >= 0.5).mean():.4%}")
        print(f"\nMean AUC: {np.nanmean(list(auc_results.values())):.6f}")
    else:
        missing = [c for c in TARGET_COLUMNS if c not in test_df.columns]
        print(f"\nMissing target columns in test.pkl: {missing} — skipping AUC.")

    display_cols = ["sess_seq_len"] + [f"logit_{c}" for c in TARGET_COLUMNS] + [f"prob_{c}" for c in TARGET_COLUMNS]

else:
    # Single target (original behavior)
    inference_df["logit"] = all_logits_out
    inference_df["prob"] = all_probs

    if TARGET_COLUMNS[0] in test_df.columns:
        gt = test_df[TARGET_COLUMNS[0]].values.astype(np.float32)
        try:
            auc = roc_auc_score(gt, all_probs)
            print(f"\nAUC: {auc:.6f}")
        except ValueError:
            print("\nAUC: could not compute (single class?)")
        print(f"Positive rate (ground truth): {gt.mean():.4%}")
        print(f"Positive rate (pred > 0.5):   {(all_probs >= 0.5).mean():.4%}")
    else:
        print(f"\nNo '{TARGET_COLUMNS[0]}' column in test.pkl — skipping AUC.")

    display_cols = ["sess_seq_len", "logit", "prob"]

print(f"\nInference DataFrame shape: {inference_df.shape}")
print(inference_df[display_cols].head(10))

inference_df.to_pickle("inference_results.pkl")
print(f"\nSaved inference results to inference_results.pkl")